In [1]:
# HuggingFace Login
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HUGGING_FACE_TOKEN")
login(hf_token)

In [2]:
import pandas as pd
from datasets import Dataset, ClassLabel
import numpy as np
import random
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from transformers import Trainer
from transformers import EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import DataLoader
import gc
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import pipeline
from datasets import concatenate_datasets

In [ ]:
ZERO_SHOT_MODELS = [
    "cross-encoder/nli-MiniLM2-L6-H768",
    "typeform/distilbert-base-uncased-mnli",
    "MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli-ling-binary",
    "tasksource/deberta-small-long-nli",
    "MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33",
    "cmarkea/distilcamembert-base-nli",
    "MoritzLaurer/xtremedistil-l6-h256-zeroshot-v1.1-all-33",
    "MoritzLaurer/deberta-v3-base-zeroshot-v1"
]


LABELS = ["met palliative care needs", "unmet palliative care needs"]
LABEL_TO_NUM = {"met palliative care needs": 0, "unmet palliative care needs": 1}
LABEL_EXPANSIONS = {
    "met palliative care needs": [
        "palliative care needs are adequately met",
        "symptoms and care needs are well managed",
        "the patient is receiving appropriate palliative care",
        "care needs are sufficiently addressed"
    ],
    "unmet palliative care needs": [
        "palliative care needs are not adequately met",
        "symptoms or care needs are poorly managed",
        "the patient requires additional palliative care support",
        "care needs are not sufficiently addressed"
    ]
}

HYPOTHESIS_TEMPLATES = [
    # Generic hypothesis templates.
    "This example is about {}.",
    # Generic medical hypothesis templates. 
    "The patient's palliative care needs are {}.",
    "Overall, the patient's care needs are {}.",
    "From this note, it can be inferred that {}.",
    "This clinical note indicates that the patient's care needs are {}.",
    "Based on this note, the patient's care needs are {}.",
    # Clinical language.
    "The patient's condition suggests that {}.",
    "This note suggests that the patient is experiencing {}.",
    "The patient's current situation reflects {}.",
    # Care quality framing.
    "The patient's symptoms and care needs are {}.",
    "The patient's care is {}.",
    # Documentation style focus.
    "This note indicates a situation where {}.",
    "The note documents that {}.",
    "The record indicates that {}.",
    "The clinical documentation suggests that {}.",
]

In [4]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [5]:
# Load the dataset
df = pd.read_csv("./dataSyntheticAll.csv")
dataset = Dataset.from_pandas(df)

# Map dataset labels to 1s or 0s.
label_map = {
    "met palliative care needs": 0,
    "unmet palliative care needs": 1
}

dataset = dataset.map(lambda x: {"label": label_map[f"{x["needs"]} palliative care needs"]})


label_feature = ClassLabel(names=["met palliative care needs", "unmet palliative care needs"])
dataset = dataset.cast_column("label", label_feature)

Map:   0%|          | 0/5783 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/5783 [00:00<?, ? examples/s]

In [6]:
# Split the dataset. (80/10/10)
dataset = dataset.train_test_split(test_size=0.2, seed=RANDOM_STATE, stratify_by_column="label")

train_dataset = dataset["train"]
temp_dataset = dataset["test"]

temp_split = temp_dataset.train_test_split(test_size=0.5, seed=RANDOM_STATE, stratify_by_column="label")

val_dataset = temp_split["train"]
test_dataset = temp_split["test"]

all_dataset = concatenate_datasets([train_dataset, val_dataset, test_dataset])

# Check distributions.
def check_distribution(dataset, name):
    df = dataset.to_pandas()
    counts = df["label"].value_counts(normalize=True)
    print(f"{name} distribution:")
    print(counts)

check_distribution(train_dataset, "Train")
check_distribution(val_dataset, "Validation")
check_distribution(test_dataset, "Test")
check_distribution(all_dataset, "All Data")

Train distribution:
label
0    0.502162
1    0.497838
Name: proportion, dtype: float64
Validation distribution:
label
0    0.50173
1    0.49827
Name: proportion, dtype: float64
Test distribution:
label
0    0.502591
1    0.497409
Name: proportion, dtype: float64
All Data distribution:
label
0    0.502162
1    0.497838
Name: proportion, dtype: float64


In [7]:
# Make evaluation function.
def evaluate_model(preds, labels):
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [8]:
results = {}
device = 0 if torch.cuda.is_available() else -1

def run_models(dataset, hypo_temp="This clinical note indicates that {}."):
    for model_name in ZERO_SHOT_MODELS:
        print(f"\nRunning zero-shot model: {model_name}")

        classifier = pipeline(
            "zero-shot-classification",
            model=model_name,
            device=device,
        )

        texts = [ex["report"] for ex in dataset]
        true_labels = [ex["label"] for ex in dataset]

        # Flatten expanded labels.
        flat_labels = []
        label_map = {}

        for base_label, expansions in LABEL_EXPANSIONS.items():
            for exp in expansions:
                flat_labels.append(exp)
                label_map[exp] = base_label

        # Run model.
        outputs = classifier(
            texts,
            candidate_labels=flat_labels,
            hypothesis_template=hypo_temp,
            batch_size=16
        )

        preds = []

        # Aggregate scores per base label.
        for o in outputs:
            scores_by_class = {k: 0.0 for k in LABEL_EXPANSIONS.keys()}

            for label, score in zip(o["labels"], o["scores"]):
                base_label = label_map[label]
                scores_by_class[base_label] += score

            # Pick best aggregated label.
            pred = max(scores_by_class, key=scores_by_class.get)
            preds.append(pred)
        numeric_preds = [0 if pred == "met palliative care needs" else 1 for pred in preds]
        metrics = evaluate_model(numeric_preds, true_labels)

        # Inner dictionary creation.
        if model_name not in results:
            results[model_name] = {}  

        results[model_name][hypo_temp] = metrics

        # Output.
        print("----------------------------------------------")
        print("--------------- Metric Results ---------------")
        print(f"\nModel: {model_name} \nHypothesis Template: {hypo_temp}")
        for k, v in metrics.items():
            print(f"{k}: {v:.4f}")

        print("--------------- Confusion Matrix ---------------")
        print(confusion_matrix(true_labels, numeric_preds))

        print("--------------- Classification Report ---------------")
        print(classification_report(true_labels, numeric_preds))

        print("----------------------------------------------\n\n\n")

        # Free memory.
        del classifier
        torch.cuda.empty_cache()
        gc.collect()

    return results


In [9]:
# Run all hypothesis templates.
test_results = []
for template in HYPOTHESIS_TEMPLATES:
    test_results.append(run_models(test_dataset, hypo_temp=template))

# Save in DataFrame.
rows = []

for model_name, templates in results.items():
    for hypo_template, metrics in templates.items():
        row = {"model": model_name, "hypothesis_template": hypo_template}
        row.update(metrics)
        rows.append(row)

# Save DataFrame.
test_df = pd.DataFrame(rows)
test_df.to_csv('./test_data_all_zero_shot_results.csv', index=False)


Running zero-shot model: intfloat/e5-base-v2


config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

----------------------------------------------
--------------- Metric Results ---------------

Model: intfloat/e5-base-v2 
Hypothesis Template: The record indicates that {}.
accuracy: 0.5009
precision: 0.4991
recall: 1.0000
f1: 0.6659
--------------- Confusion Matrix ---------------
[[  2 289]
 [  0 288]]
--------------- Classification Report ---------------
              precision    recall  f1-score   support

           0       1.00      0.01      0.01       291
           1       0.50      1.00      0.67       288

    accuracy                           0.50       579
   macro avg       0.75      0.50      0.34       579
weighted avg       0.75      0.50      0.34       579

----------------------------------------------




Running zero-shot model: intfloat/e5-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Failed to determine 'entailment' label id from the label2id mapping in the model config. Setting to -1. Define a descriptive label2id mapping in the model config to ensure correct outputs.


----------------------------------------------
--------------- Metric Results ---------------

Model: intfloat/e5-base-v2 
Hypothesis Template: The clinical documentation suggests that {}.
accuracy: 0.5026
precision: 0.0000
recall: 0.0000
f1: 0.0000
--------------- Confusion Matrix ---------------
[[291   0]
 [288   0]]
--------------- Classification Report ---------------
              precision    recall  f1-score   support

           0       0.50      1.00      0.67       291
           1       0.00      0.00      0.00       288

    accuracy                           0.50       579
   macro avg       0.25      0.50      0.33       579
weighted avg       0.25      0.50      0.34       579

----------------------------------------------





/home/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.cap

In [10]:
test_df.sort_values(by='accuracy', ascending=False)

,model,hypothesis_template,accuracy,precision,recall,f1
3,intfloat/e5-base-v2,"From this note, it can be inferred that {}.",0.690846,0.779487,0.527778,0.629400
7,intfloat/e5-base-v2,This note suggests that the patient is experie...,0.611399,0.698113,0.385417,0.496644
5,intfloat/e5-base-v2,"Based on this note, the patient's care needs a...",0.573402,0.550369,0.777778,0.644604
11,intfloat/e5-base-v2,This note indicates a situation where {}.,0.537133,0.518657,0.965278,0.674757
0,intfloat/e5-base-v2,This example is about {}.,0.514680,0.818182,0.031250,0.060201
6,intfloat/e5-base-v2,The patient's condition suggests that {}.,0.514680,0.888889,0.027778,0.053872
2,intfloat/e5-base-v2,"Overall, the patient's care needs are {}.",0.507772,0.540541,0.069444,0.123077
1,intfloat/e5-base-v2,The patient's palliative care needs are {}.,0.506045,0.541667,0.045139,0.083333
14,intfloat/e5-base-v2,The clinical documentation suggests that {}.,0.502591,0.000000,0.000000,0.000000
13,intfloat/e5-base-v2,The record indicates that {}.,0.500864,0.499133,1.000000,0.665896


In [11]:
test_df.sort_values(by='precision', ascending=False)

,model,hypothesis_template,accuracy,precision,recall,f1
6,intfloat/e5-base-v2,The patient's condition suggests that {}.,0.514680,0.888889,0.027778,0.053872
0,intfloat/e5-base-v2,This example is about {}.,0.514680,0.818182,0.031250,0.060201
3,intfloat/e5-base-v2,"From this note, it can be inferred that {}.",0.690846,0.779487,0.527778,0.629400
7,intfloat/e5-base-v2,This note suggests that the patient is experie...,0.611399,0.698113,0.385417,0.496644
5,intfloat/e5-base-v2,"Based on this note, the patient's care needs a...",0.573402,0.550369,0.777778,0.644604
1,intfloat/e5-base-v2,The patient's palliative care needs are {}.,0.506045,0.541667,0.045139,0.083333
2,intfloat/e5-base-v2,"Overall, the patient's care needs are {}.",0.507772,0.540541,0.069444,0.123077
11,intfloat/e5-base-v2,This note indicates a situation where {}.,0.537133,0.518657,0.965278,0.674757
13,intfloat/e5-base-v2,The record indicates that {}.,0.500864,0.499133,1.000000,0.665896
10,intfloat/e5-base-v2,The patient's care is {}.,0.495682,0.496540,0.996528,0.662818


In [12]:
test_df.sort_values(by='recall', ascending=False)

,model,hypothesis_template,accuracy,precision,recall,f1
13,intfloat/e5-base-v2,The record indicates that {}.,0.500864,0.499133,1.000000,0.665896
10,intfloat/e5-base-v2,The patient's care is {}.,0.495682,0.496540,0.996528,0.662818
12,intfloat/e5-base-v2,The note documents that {}.,0.488774,0.492933,0.968750,0.653396
11,intfloat/e5-base-v2,This note indicates a situation where {}.,0.537133,0.518657,0.965278,0.674757
5,intfloat/e5-base-v2,"Based on this note, the patient's care needs a...",0.573402,0.550369,0.777778,0.644604
8,intfloat/e5-base-v2,The patient's current situation reflects {}.,0.409326,0.446215,0.777778,0.567089
3,intfloat/e5-base-v2,"From this note, it can be inferred that {}.",0.690846,0.779487,0.527778,0.629400
7,intfloat/e5-base-v2,This note suggests that the patient is experie...,0.611399,0.698113,0.385417,0.496644
9,intfloat/e5-base-v2,The patient's symptoms and care needs are {}.,0.333333,0.339869,0.361111,0.350168
2,intfloat/e5-base-v2,"Overall, the patient's care needs are {}.",0.507772,0.540541,0.069444,0.123077


In [13]:
test_df.sort_values(by='f1', ascending=False)

,model,hypothesis_template,accuracy,precision,recall,f1
11,intfloat/e5-base-v2,This note indicates a situation where {}.,0.537133,0.518657,0.965278,0.674757
13,intfloat/e5-base-v2,The record indicates that {}.,0.500864,0.499133,1.000000,0.665896
10,intfloat/e5-base-v2,The patient's care is {}.,0.495682,0.496540,0.996528,0.662818
12,intfloat/e5-base-v2,The note documents that {}.,0.488774,0.492933,0.968750,0.653396
5,intfloat/e5-base-v2,"Based on this note, the patient's care needs a...",0.573402,0.550369,0.777778,0.644604
3,intfloat/e5-base-v2,"From this note, it can be inferred that {}.",0.690846,0.779487,0.527778,0.629400
8,intfloat/e5-base-v2,The patient's current situation reflects {}.,0.409326,0.446215,0.777778,0.567089
7,intfloat/e5-base-v2,This note suggests that the patient is experie...,0.611399,0.698113,0.385417,0.496644
9,intfloat/e5-base-v2,The patient's symptoms and care needs are {}.,0.333333,0.339869,0.361111,0.350168
2,intfloat/e5-base-v2,"Overall, the patient's care needs are {}.",0.507772,0.540541,0.069444,0.123077
